# 04 - Database Layer and Multi-table Relations

> **When to use**: When you have multiple interrelated tables and need to ensure foreign key integrity.
>
> **Core concept**: sqlseed auto-detects table dependencies, fills in topological order, and shares values across tables via SharedPool.

## Use Cases

- Multiple tables with FOREIGN KEY constraints → sqlseed auto-orders fills
- Two tables share same column name (e.g., `member_no`) → SharedPool implicit association
- Different column names but need association (e.g., `department_id` → `id`) → ColumnAssociation explicit association
- Large data volume write performance optimization → Pragma three-level optimization

## What You Will Learn

- Dual adapter architecture (SQLAlchemyAdapter / RawSQLiteAdapter)
- Pragma three-level write optimization
- SharedPool cross-table value sharing
- ColumnAssociation explicit association
- BLOB / large text handling

See architecture.zh-CN.md §5

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Flow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| **→ 04** | **Database Layer and Multi-table Relations** | **Database + Core** | **01** |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| 07 | AI Smart Config | Plugins: AI | 01 |
| 08 | MCP Server Integration | Plugins: MCP | 07 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Test Integration Patterns | Testing | 01 |

---

In [1]:
from pathlib import Path

# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]

Generating members:   0%|          | 0/20 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/10 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/8 [00:00<?, ?it/s]

sqlseed 0.1.16.dev1+g0824e8553.d20260505 | Database: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db


### 📍 Architecture Location

| Module | File | Core Class/Function |
|------|------|------------|
| Relation resolution | `src/sqlseed/core/relation.py` | `RelationResolver` |
| Database interface | `src/sqlseed/database/_protocol.py` | `DatabaseAdapter` |

> See architecture diagram: [§5 Database Layer Architecture](../docs/architecture.zh-CN.md#5-数据库层架构)

## 1. See It in Action — Multi-table Relation Auto-ordering

Database has foreign key constraints? sqlseed auto-detects table dependencies and fills in topological order — **no need to specify order manually**:

In [2]:
from sqlseed import fill_from_config
from sqlseed.config.loader import save_config
from sqlseed.config.models import GeneratorConfig, TableConfig

# Fill 5 tables at once — sqlseed auto-handles FK order
config = GeneratorConfig(
    db_path=str(db_path),
    tables=[
        TableConfig(name='organizations', count=3, clear_before=True),
        TableConfig(name='members', count=10, clear_before=True),
        TableConfig(name='projects', count=5, clear_before=True),
        TableConfig(name='tasks', count=20, clear_before=True),
        TableConfig(name='tags', count=5, clear_before=True),
    ]
)
config_path = Path('_fk_demo.yaml')
save_config(config, str(config_path))

results = fill_from_config(str(config_path))
print(f"{'Table':<15s}  {'Rows':>6s}  {'Elapsed':>8s}  {'Speed':>10s}")
print('-' * 45)
for r in results:
    print(f"{r.table_name:<15s}  {r.count:>6d}  {r.elapsed:>7.3f}s  {r.rows_per_second:>8.0f} rows/s")

config_path.unlink(missing_ok=True)

Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

Generating members:   0%|          | 0/10 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/5 [00:00<?, ?it/s]

Generating tasks:   0%|          | 0/20 [00:00<?, ?it/s]

Generating tags:   0%|          | 0/5 [00:00<?, ?it/s]

Table              Rows   Elapsed      Speed
---------------------------------------------
organizations         3    0.073s        41 rows/s
members              10    0.053s       190 rows/s
projects              5    0.047s       107 rows/s
tasks                20    0.043s       468 rows/s
tags                  5    0.043s       117 rows/s


Note the output order — `organizations` before `members`, `projects` before `tasks`. sqlseed determines the correct fill order via topological sort, ensuring foreign key references are valid.

Below we break down each mechanism of the database layer in detail.

## 2. Dual Adapter Architecture

sqlseed supports two database adapters:

| Adapter | Dependency | Default | Characteristics |
|--------|------|:----:|------|
| `SQLAlchemyAdapter` | SQLAlchemy | ✅ | Feature-rich, multi-DB support |
| `RawSQLiteAdapter` | None (stdlib) | - | Zero-dep, test-only |

sqlseed auto-selects: `SQLAlchemyAdapter` is the required default for production (multi-DB support); `RawSQLiteAdapter` is a test-only zero-dependency fallback.

In [3]:
from sqlseed.database import SQLAlchemyAdapter, RawSQLiteAdapter

# SQLAlchemyAdapter is the required production adapter (multi-DB support).
# RawSQLiteAdapter is a test-only, zero-dependency fallback.
print(f"SQLAlchemy available: True")
print(f"Active adapter: SQLAlchemyAdapter")

SQLAlchemy available: True
Active adapter: SQLAlchemyAdapter


## 3. Pragma Optimization

sqlseed auto-optimizes SQLite Pragma settings during batch writes to improve performance:

| Level | journal_mode | synchronous | locking_mode | Use Case |
|------|-------------|-------------|-------------|----------|
| light | WAL | NORMAL | - | Small data volume (<1K rows) |
| moderate | WAL | OFF | - | Medium data volume (1K-10K rows) |
| aggressive | MEMORY | OFF | EXCLUSIVE | Large data volume (>10K rows) |

Enabled via `optimize_pragma=True` (default); original settings are auto-restored on exit.

In [4]:

# Pragma optimization auto-tunes journal_mode, synchronous, etc. during batch writes
# Effect is more visible with large data volumes; small volumes are affected by UNIQUE constraint solving overhead
result_no_opt = fill(str(db_path), table="tasks", count=3000, optimize_pragma=False, clear_before=True)
print(f"Without optimization: {result_no_opt.count} rows in {result_no_opt.elapsed:.3f}s ({result_no_opt.rows_per_second:.0f} rows/s)")  # noqa: E501

result = fill(str(db_path), table="tasks", count=3000, optimize_pragma=True, clear_before=True)
print(f"With Pragma optimization: {result.count} rows in {result.elapsed:.3f}s ({result.rows_per_second:.0f} rows/s)")

Generating tasks:   0%|          | 0/3000 [00:00<?, ?it/s]

Without optimization: 3000 rows in 0.433s (6927 rows/s)


Generating tasks:   0%|          | 0/3000 [00:00<?, ?it/s]

With Pragma optimization: 3000 rows in 0.628s (4775 rows/s)


## 4. FK Resolution: Explicit vs Implicit

### Explicit FK

FOREIGN KEY constraints declared in CREATE TABLE:

```sql
CREATE TABLE tasks (
    project_id INTEGER NOT NULL,
    FOREIGN KEY (project_id) REFERENCES projects(project_id)
);
```

sqlseed auto-detects explicit FKs and generates data referencing existing parent table values.

### Implicit FK

No FOREIGN KEY declared, but column name matches parent table primary key (e.g., `member_id` in `reviews` table). sqlseed auto-infers association via name matching.

In [5]:
import sqlite3

conn = sqlite3.connect(str(db_path))

print("--- reviews table FK ---")
fk_list = conn.execute("PRAGMA foreign_key_list(reviews)").fetchall()
print(f"Explicit FKs: {fk_list}")
print("Note: member_id has no explicit FK, but sqlseed detects it via name matching")

conn.close()

--- reviews table FK ---
Explicit FKs: [(0, 0, 'tasks', 'task_id', 'task_id', 'NO ACTION', 'NO ACTION', 'NONE')]
Note: member_id has no explicit FK, but sqlseed detects it via name matching


## 5. SharedPool Cross-table Value Sharing

When multiple tables reference the same parent table, `SharedPool` ensures reference consistency:

- `tasks.project_id` references `projects.project_id` pool (explicit FK)
- `tasks.assignee_id` references `members.member_id` pool (explicit FK)
- `reviews.task_id` references `tasks.task_id` pool (explicit FK)
- `reviews.member_id` references `members.member_id` pool (implicit FK, name matching)

See architecture.md §5 SharedPool

In [6]:

with connect(str(db_path)) as orch:
    r1 = orch.fill_table("organizations", count=3)
    r2 = orch.fill_table("members", count=10)
    r3 = orch.fill_table("projects", count=5)
    r4 = orch.fill_table("tasks", count=20)
    r5 = orch.fill_table("reviews", count=10)

conn = sqlite3.connect(str(db_path))
project_ids_tasks = {r[0] for r in conn.execute("SELECT DISTINCT project_id FROM tasks").fetchall()}
project_ids_projects = {r[0] for r in conn.execute("SELECT project_id FROM projects").fetchall()}
print(f"project_ids in tasks: {project_ids_tasks}")
print(f"project_ids in projects: {project_ids_projects}")
print(f"All task project_ids exist in projects: {project_ids_tasks.issubset(project_ids_projects)}")
conn.close()

Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

Generating members:   0%|          | 0/10 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/5 [00:00<?, ?it/s]

Generating tasks:   0%|          | 0/20 [00:00<?, ?it/s]

Generating reviews:   0%|          | 0/10 [00:00<?, ?it/s]

project_ids in tasks: {1, 2, 3, 4, 5, 6, 8, 9, 10}
project_ids in projects: {1, 2, 3, 4, 5, 6, 7, 8, 9, 10}
All task project_ids exist in projects: True


## 6. ColumnAssociation Explicit Association

When implicit FK cannot be inferred, use `ColumnAssociation` to explicitly declare cross-table associations:

```yaml
associations:
  - column: member_no
    ref_table: members
    ref_column: member_no
```

See 09-config-deep-dive.ipynb

## 7. Multi-table Batch Filling

Use `fill_from_config` to batch-fill multiple tables from YAML/JSON config, auto-ordered by FK dependencies:

In [7]:
from sqlseed import GeneratorConfig, ProviderType, TableConfig, fill_from_config
from sqlseed.config.loader import save_config

config = GeneratorConfig(
    db_path=str(db_path),
    provider=ProviderType("mimesis"),
    tables=[
        TableConfig(name="organizations", count=3),
        TableConfig(name="members", count=10),
        TableConfig(name="projects", count=5),
        TableConfig(name="tasks", count=30),
    ],
)

config_path = str(Path("batch_config.yaml"))
save_config(config, config_path)

results = fill_from_config(config_path, clear_before=True)
for r in results:
    print(f"{r.table_name}: {r.count} rows in {r.elapsed:.3f}s")

Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]

Generating members:   0%|          | 0/10 [00:00<?, ?it/s]

Generating projects:   0%|          | 0/5 [00:00<?, ?it/s]

Generating tasks:   0%|          | 0/30 [00:00<?, ?it/s]

organizations: 3 rows in 0.056s
members: 10 rows in 0.088s
projects: 5 rows in 0.029s
tasks: 30 rows in 0.040s


## 8. clear_before and FK Constraints

`clear_before=True` DELETEs then INSERTs. Note: if child tables reference parent table data, clear child tables first, then parent tables.

sqlseed's `fill_from_config` handles this automatically via topological sort, but when using `fill()` standalone, pay attention to order manually.

## 9. BLOB Column Handling

BLOB-typed columns generate random binary data:

In [8]:

# file_name matches *_name pattern -> name generator (generates person names)
# Here we override with columns for a more sensible filename format
rows = preview(str(db_path), table="attachments", count=2,
               columns={"file_name": {"type": "pattern", "regex": "[a-z]{8}[.](pdf|png|docx)"}})
for row in rows:
    print(f"file_name={row['file_name']}, uploaded_at={row['uploaded_at']}")
print("\nfile_data (BLOB, nullable) and file_size (DEFAULT 0) are skipped by the strategy chain")
print("BLOB columns are skipped by default in nullable strategy; columns with DEFAULT use the default value")

file_name=djknrqdb.docx, uploaded_at=2019-01-03 18:09:44.650555
file_name=svihpyfk.png, uploaded_at=2021-06-15 18:08:02.745692

file_data (BLOB, nullable) and file_size (DEFAULT 0) are skipped by the strategy chain
BLOB columns are skipped by default in nullable strategy; columns with DEFAULT use the default value


## 📋 DatabaseAdapter Full API

DatabaseAdapter Protocol defines all database operation interfaces:

In [9]:
with sqlseed.connect(str(db_path)) as orch:
    print('=== get_table_names() ===')
    tables = orch.get_table_names()
    print(f'  Tables: {tables}')

    print('\n=== get_column_info(organizations) ===')
    col_info = orch.get_column_info('organizations')
    for col in col_info:
        print(f'  {col.name}: type={col.type}, pk={col.is_primary_key}')

    print('\n=== get_foreign_keys(tasks) ===')
    fks = orch.get_foreign_keys('tasks')
    for fk in fks:
        print(f'  {fk}')

    print('\n=== get_row_count() ===')
    for table in tables:
        count = orch.get_row_count(table)
        print(f'  {table}: {count} rows')

=== get_table_names() ===
  Tables: ['organizations', 'members', 'sqlite_sequence', 'projects', 'tasks', 'reviews', 'tags', 'task_tags', 'attachments']

=== get_column_info(organizations) ===
  org_code: type=VARCHAR(16), pk=True
  name: type=VARCHAR(64), pk=False
  parent_code: type=VARCHAR(16), pk=False
  description: type=TEXT, pk=False
  is_active: type=INTEGER, pk=False
  member_count: type=INTEGER, pk=False
  created_at: type=TEXT, pk=False

=== get_foreign_keys(tasks) ===
  ForeignKeyInfo(column='assignee_id', ref_table='members', ref_column='member_id')
  ForeignKeyInfo(column='project_id', ref_table='projects', ref_column='project_id')

=== get_row_count() ===
  organizations: 3 rows
  members: 10 rows
  sqlite_sequence: 5 rows
  projects: 5 rows
  tasks: 30 rows
  reviews: 10 rows
  tags: 5 rows
  task_tags: 0 rows
  attachments: 0 rows


## ⚡ PragmaOptimizer Three-level Optimization Parameters

PragmaOptimizer auto-selects optimization level based on expected row count:

| Level | Trigger | Key PRAGMA |
|---|---|---|
| Light | < 1,000 rows | journal_mode=WAL, synchronous=NORMAL |
| Moderate | 1,000-10,000 rows | + cache_size=-32000, temp_store=MEMORY |
| Aggressive | > 10,000 rows | + mmap_size=512MB, locking_mode=EXCLUSIVE |

Original PRAGMA settings are auto-restored after execution.

In [10]:

print("PragmaOptimizer three-level optimization parameters:")
print("  Light threshold: < 1,000 rows")
print("  Moderate threshold: 1,000 - 10,000 rows")
print("  Aggressive threshold: > 10,000 rows")

with sqlseed.connect(str(db_path), optimize_pragma=True) as orch:
    result = orch.fill_table("organizations", count=5, clear_before=True)
    print(f"\nFilled {result.count} rows (Light level)")

PragmaOptimizer three-level optimization parameters:
  Light threshold: < 1,000 rows
  Moderate threshold: 1,000 - 10,000 rows
  Aggressive threshold: > 10,000 rows


Generating organizations:   0%|          | 0/5 [00:00<?, ?it/s]


Filled 5 rows (Light level)


## 10. Summary

| Feature | Description |
|------|------|
| Dual adapter | SQLAlchemyAdapter (default) / RawSQLiteAdapter (test-only) |
| Pragma optimization | Three-level strategy, auto-selected |
| Explicit FK | Declared in CREATE TABLE |
| Implicit FK | Auto-inferred when column name matches parent PK |
| SharedPool | Cross-table reference consistency |
| ColumnAssociation | Explicit association declaration |
| fill_from_config | Auto topological sort, batch fill |
| BLOB | Random binary data |

**Next**: [05-dag-and-constraints.ipynb](05-dag-and-constraints.ipynb) — DAG topological sort and constraint solving

In [11]:
# ✅ Verification: ensure data was successfully generated and written
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # Basic row count verification
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
